# Vignette 06: Tool Comparison — mlsurv vs. scikit-survival and lifelines

This notebook compares mlsurv with two widely used Python survival libraries: **scikit-survival**, which mlsurv wraps for its CoxPH and random survival forest (RSF) models, and **lifelines**, an independent CoxPH implementation. It asks three questions:

1. **Code required** — how much code does each tool need for the same workflow?
2. **Fidelity** — given the same data, split, and settings, does mlsurv reproduce scikit-survival's results?
3. **Tuning** — does mlsurv's automatic hyperparameter tuning (on by default) change test-set performance, and does it need extra code?

Every tool runs the same workflow: an event-stratified 80/20 train/test split, 5-fold cross-validation (CV) on the training set, a final fit on the full training set, and Harrell's C-index on the **held-out test set** with a 95% bootstrap confidence interval. All comparisons use test-set C-indices: a tuned model's CV score is optimistic, because tuning selects the settings that score best on those same folds.

**Datasets**: GBSG2 (Schumacher et al. 1994; n=686, breast cancer) and METABRIC (Curtis et al. 2012 / Katzman et al. 2018; n=1903, breast cancer).

*Runtime: ~6 minutes on a modern laptop.*

In [ ]:
import ast, inspect, textwrap, warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')
warnings.filterwarnings('ignore', category=UserWarning, module='lifelines')

import logging
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sksurv.datasets import load_gbsg2
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from lifelines import CoxPHFitter
from lifelines.utils import k_fold_cross_validation

from mlsurv import SurvivalLearner
from mlsurv.utils import load_survival_data, to_structured_array

logging.getLogger('mlsurv').setLevel(logging.WARNING)  # keep output readable

RANDOM_STATE = 42
N_BOOT = 1000

In [ ]:
# Library versions used for this run
from importlib.metadata import version
for dist in ['mlsurv', 'scikit-survival', 'lifelines', 'scikit-learn']:
    print(f'{dist}: {version(dist)}')

In [ ]:
# GBSG2 (bundled with scikit-survival): encode the three categorical columns
X_gbsg2, y_gbsg2 = load_gbsg2()
X_gbsg2 = X_gbsg2.assign(
    horTh=(X_gbsg2['horTh'] == 'yes').astype(int),
    menostat=(X_gbsg2['menostat'] == 'Post').astype(int),
    tgrade=X_gbsg2['tgrade'].map({'I': 1, 'II': 2, 'III': 3}).astype(int),
)
y_gbsg2 = to_structured_array(y_gbsg2)

# METABRIC (local CSV)
X_metabric, y_metabric = load_survival_data('data/metabric.csv', time_col='duration', event_col='event')

DATASETS = {'GBSG2': (X_gbsg2, y_gbsg2), 'METABRIC': (X_metabric, y_metabric)}
for name, (X, y) in DATASETS.items():
    print(f'{name}: n={len(X)}, features={X.shape[1]}, events={y["event"].sum()}')

---
## Section 1: The same workflow in each tool

Each function below is the code a user writes for the workflow described above: split, 5-fold CV, final fit, test-set C-index, and a bootstrap confidence interval. Each uses the library's own helpers where they exist (scikit-learn's `cross_val_score` for scikit-survival, `k_fold_cross_validation` for lifelines); neither library provides a bootstrap helper, so those loops are written by hand. mlsurv also runs steps the other two do not (dataset diagnostics, permutation importance, and additional metrics).

Two choices to note. The scikit-survival CoxPH model is passed in already wrapped with a scaler (Section 2), so that scaling is not counted. The lifelines workflow skips scaling, which does not change an unpenalized Cox model's ranking of patients.

In [ ]:
def mlsurv_workflow(X, y, model, **run_options):
    learner = SurvivalLearner(X, y)
    learner.setup(test_fraction=0.2, cv_folds=5, random_state=RANDOM_STATE, n_jobs=1)
    result = learner.run(models=[model], n_bootstrap=N_BOOT, verbose=False, **run_options)[model]
    return result

In [ ]:
def sksurv_workflow(X, y, model):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y['event'])
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=folds.split(X_tr, y_tr['event']))
    model.fit(X_tr, y_tr)
    risk_test = model.predict(X_te)
    test_c = concordance_index_censored(y_te['event'], y_te['time'], risk_test)[0]
    rng = np.random.RandomState(RANDOM_STATE)
    boot = []
    for _ in range(N_BOOT):
        idx = rng.choice(len(y_te), size=len(y_te), replace=True)
        boot.append(concordance_index_censored(y_te[idx]['event'], y_te[idx]['time'], risk_test[idx])[0])
    ci = np.percentile(boot, [2.5, 97.5])
    return dict(cv=np.mean(cv_scores), test=test_c, ci=ci, risk=risk_test)

In [ ]:
def lifelines_workflow(X, y):
    df = X.assign(duration=y['time'], event=y['event'])
    df_tr, df_te = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=df['event'])
    cv_scores = k_fold_cross_validation(CoxPHFitter(), df_tr, duration_col='duration', event_col='event',
                                        k=5, scoring_method='concordance_index', seed=RANDOM_STATE)
    cph = CoxPHFitter().fit(df_tr, duration_col='duration', event_col='event')
    test_c = cph.score(df_te, scoring_method='concordance_index')
    rng = np.random.RandomState(RANDOM_STATE)
    boot = []
    for _ in range(N_BOOT):
        idx = rng.choice(len(df_te), size=len(df_te), replace=True)
        boot.append(cph.score(df_te.iloc[idx], scoring_method='concordance_index'))
    ci = np.percentile(boot, [2.5, 97.5])
    return dict(cv=np.mean(cv_scores), test=test_c, ci=ci, risk=cph.predict_partial_hazard(df_te).values)

### Code required

We count **statements** in each function body (excluding the `return`), so the count does not depend on line wrapping or formatting.

In [ ]:
def count_statements(fn):
    body = ast.parse(textwrap.dedent(inspect.getsource(fn))).body[0].body
    return sum(isinstance(node, ast.stmt)
               for stmt in body if not isinstance(stmt, ast.Return)
               for node in ast.walk(stmt))

loc = pd.Series({tool: count_statements(fn) for tool, fn in
                 [('mlsurv', mlsurv_workflow), ('scikit-survival', sksurv_workflow),
                  ('lifelines', lifelines_workflow)]}, name='statements')
loc.to_frame()

---
## Section 2: Fidelity — same settings, same results

We run CoxPH in all three tools, and an RSF in mlsurv and scikit-survival. For the RSF, mlsurv is given scikit-survival's default hyperparameters and no feature scaling (scikit-survival's RSF is fit on unscaled features), so the two runs differ only in which package executes them.

The helper below (not part of any workflow) recovers the test split so mlsurv's test-set predictions can be compared one-to-one with scikit-survival's. It checks that the split matches: mlsurv's reported test C-index must equal the C-index computed from those predictions.

In [ ]:
SKSURV_RSF_DEFAULTS = dict(n_estimators=100, min_samples_split=6, min_samples_leaf=3, max_features='sqrt')

def summarize_mlsurv(result, X, y):
    # Recover the same 80/20 split, then score mlsurv's test-set predictions.
    _, X_te, _, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y['event'])
    risk = result.model.predict(X_te)
    c = concordance_index_censored(y_te['event'], y_te['time'], risk)[0]
    assert np.isclose(c, result.evaluation.c_index_test), 'test split does not match'
    boot = result.evaluation.bootstrap.summary.set_index('metric').loc['harrell']
    return dict(cv=result.cv_metric_mean, test=c, ci=(boot['ci_lower'], boot['ci_upper']), risk=risk)

fidelity, rsf_sksurv = [], {}
for name, (X, y) in DATASETS.items():
    runs = {
        ('CoxPH', 'mlsurv'): summarize_mlsurv(mlsurv_workflow(X, y, 'coxph', tune=False), X, y),
        ('CoxPH', 'scikit-survival'): sksurv_workflow(X, y, make_pipeline(StandardScaler(), CoxPHSurvivalAnalysis())),
        ('CoxPH', 'lifelines'): lifelines_workflow(X, y),
        ('RSF', 'mlsurv'): summarize_mlsurv(mlsurv_workflow(
            X, y, 'rsf', tune=False, model_params={'rsf': SKSURV_RSF_DEFAULTS},
            preprocessing={'rsf': {'scaler': None}}), X, y),
        ('RSF', 'scikit-survival'): sksurv_workflow(X, y, RandomSurvivalForest(random_state=RANDOM_STATE)),
    }
    rsf_sksurv[name] = runs[('RSF', 'scikit-survival')]
    for model in ['CoxPH', 'RSF']:
        ref = runs[(model, 'scikit-survival')]['risk']
        for (m, tool), r in runs.items():
            if m == model:
                fidelity.append(dict(dataset=name, model=model, tool=tool, test_c=r['test'],
                                     ci=f"[{r['ci'][0]:.3f}, {r['ci'][1]:.3f}]",
                                     same_predictions_as_sksurv=None if tool == 'lifelines'
                                     else bool(np.allclose(r['risk'], ref, rtol=1e-6, atol=1e-9))))

pd.DataFrame(fidelity).round(4)

**Reading the table.** mlsurv and scikit-survival produce the same test-set C-index, and `same_predictions_as_sksurv` confirms that mlsurv's risk score for every test patient matches scikit-survival's. lifelines fits the Cox model with a different tie-handling method (Efron rather than Breslow), so its C-index can differ in the last decimal places. All three tools compute the 95% interval the same way (percentile bootstrap over resampled test patients, with the fitted model held fixed), so interval differences reflect only each tool's own random resamples.

---
## Section 3: Tuning — same code, tuned models

mlsurv tunes hyperparameters by default (Optuna TPE; the number of trials is set per model, 50 for RSF), so the tuned workflow is the **same function call** without `tune=False`. Tuning uses only the training set; the test set is untouched until the final evaluation.

We compare three RSFs on each test set: scikit-survival's defaults (from Section 2), mlsurv's defaults (untuned), and mlsurv tuned. The table reports the observed difference in test C-index (tuned minus each alternative), with a 95% interval and p-value from a paired bootstrap over the same test patients.

By default we tune on GBSG2 only, to keep the runtime short. Add `'METABRIC'` to `TUNE_DATASETS` to reproduce the METABRIC result (about 40 minutes): tuned test C=0.639 versus 0.645 for scikit-survival's defaults (difference −0.005, bootstrap p=0.43) and 0.634 for mlsurv's defaults (+0.005, p=0.39), with the library versions printed above.

In [ ]:
TUNE_DATASETS = ['GBSG2']  # add 'METABRIC' to reproduce the METABRIC result (~40 minutes)

def paired_bootstrap(y_te, risk_a, risk_b, n=1000):
    # Observed test C-index difference (a - b), plus a 95% interval and p-value from shared bootstrap resamples.
    c = lambda yy, risk: concordance_index_censored(yy['event'], yy['time'], risk)[0]
    observed = c(y_te, risk_a) - c(y_te, risk_b)
    rng = np.random.RandomState(RANDOM_STATE)
    diffs = []
    for _ in range(n):
        idx = rng.choice(len(y_te), size=len(y_te), replace=True)
        diffs.append(c(y_te[idx], risk_a[idx]) - c(y_te[idx], risk_b[idx]))
    diffs = np.array(diffs)
    p = min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    return observed, np.percentile(diffs, 2.5), np.percentile(diffs, 97.5), p

tuning = []
for name in TUNE_DATASETS:
    X, y = DATASETS[name]
    _, _, _, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y['event'])
    default = summarize_mlsurv(mlsurv_workflow(X, y, 'rsf', tune=False), X, y)
    tuned_result = mlsurv_workflow(X, y, 'rsf')  # tuning is on by default
    tuned = summarize_mlsurv(tuned_result, X, y)
    print(f'{name} tuned RSF settings: {tuned_result.best_params_}')
    for label, r in [('scikit-survival defaults', rsf_sksurv[name]), ('mlsurv defaults', default), ('mlsurv tuned', tuned)]:
        row = dict(dataset=name, rsf=label, test_c=r['test'])
        if label != 'mlsurv tuned':
            d, lo, hi, p = paired_bootstrap(y_te, tuned['risk'], r['risk'])
            row.update(tuned_minus_this=d, diff_ci=f'[{lo:+.3f}, {hi:+.3f}]', bootstrap_p=p)
        tuning.append(row)

pd.DataFrame(tuning).round(4)

---
## Section 4: Feature comparison

| Feature | mlsurv | scikit-survival | lifelines |
|---------|--------|-----------------|-----------|
| CoxPH | Yes (wraps scikit-survival) | Yes | Yes |
| Penalized Cox (elastic net) | Yes (wraps scikit-survival) | Yes | Yes (`penalizer`, `l1_ratio`) |
| Random survival forest, gradient boosting | Yes (wraps scikit-survival) | Yes | No |
| XGBoost survival models | Yes | No | No |
| Deep learning (DeepSurv, DeepHit) | Yes (wraps pycox) | No | No |
| Leakage-safe preprocessing within CV folds | Default | Manual (scikit-learn `Pipeline`) | Manual |
| Cross-validation | Built-in | Via scikit-learn (`cross_val_score`) | Built-in (`k_fold_cross_validation`) |
| Model comparison on shared folds and metrics | Built-in | Manual | Manual |
| Hyperparameter tuning | Built-in, on by default (Optuna) | Via scikit-learn (e.g., `GridSearchCV`) | Via scikit-learn adapter |
| Bootstrap confidence intervals | Built-in | Manual | Manual |
| Dataset diagnostics and model recommendations | Built-in | No | Partial (`check_assumptions` for Cox) |
| External validation on new cohorts | Built-in (`validate()`) | Manual | Manual |
| Interactive HTML report | Built-in | No | No |

"Manual" means the library provides the building blocks but the user writes the workflow code, as in Section 1.

---
## Summary

1. **Code**: the workflow takes a few statements in mlsurv versus several times more in scikit-survival or lifelines, even when those libraries' own helpers are used (Section 1).
2. **Fidelity**: with the same split and settings, mlsurv reproduces scikit-survival's predictions exactly, and lifelines gives the same CoxPH test-set C-index to 4 decimal places (Section 2).
3. **Tuning**: mlsurv's tuned model needs no additional code. On GBSG2, tuning raised the RSF's test C-index from 0.688 (scikit-survival defaults) to 0.705, a gain that was not statistically significant (bootstrap p=0.16). On METABRIC, tuning did not improve on scikit-survival's defaults (this run is left out by default because it takes about 40 minutes; see Section 3). Random survival forests often perform well with default settings; models that are more sensitive to their hyperparameters, such as DeepSurv and DeepHit, benefit more from tuning (see **Vignette 03**).

For mlsurv's reproduction of published benchmark results, see **Vignette 03**.

## References

- Curtis C, Shah SP, Chin SF, et al. (2012). The genomic and transcriptomic architecture of 2,000 breast tumours reveals novel subgroups. *Nature*, 486(7403):346–352. https://doi.org/10.1038/nature10983
- Davidson-Pilon C (2019). lifelines: survival analysis in Python. *Journal of Open Source Software*, 4(40):1317. https://doi.org/10.21105/joss.01317
- Katzman JL, Shaham U, Cloninger A, Bates J, Jiang T, Kluger Y (2018). DeepSurv: personalized treatment recommender system using a Cox proportional hazards deep neural network. *BMC Medical Research Methodology*, 18:24. https://doi.org/10.1186/s12874-018-0482-1
- Pölsterl S (2020). scikit-survival: a library for time-to-event analysis built on top of scikit-learn. *Journal of Machine Learning Research*, 21(212):1–6. https://jmlr.org/papers/v21/20-729.html
- Schumacher M, Bastert G, Bojar H, et al. (1994). Randomized 2×2 trial evaluating hormonal treatment and the duration of chemotherapy in node-positive breast cancer patients (German Breast Cancer Study Group). *Journal of Clinical Oncology*, 12(10):2086–2093. https://doi.org/10.1200/JCO.1994.12.10.2086